In [ ]:
import pandas as pd
!pip install XlsxWriter
from datetime import datetime
from openpyxl import load_workbook
from openpyxl.utils import get_column_letter
import warnings

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.3/175.3 kB 10.7 MB/s eta 0:00:00


In [ ]:
CDPAR2 = pd.read_excel('CDPAR.XLSX')
DDPAR2 = pd.read_excel('DDPAR.XLSX')
LNPAR2 = pd.read_excel('LNPAR.XLSX')
ZFPSL = pd.read_excel('/content/ZFPSL_PRDCTGRP.XLSX')

T_CDPAR2 = CDPAR2[['PTYPE','PGROUP','PCURTY']]
T_DDPAR2 = DDPAR2[['SCCODE','PGRCOD','DP2CUR']]
T_LNPAR2 = LNPAR2[['PTYPE','PLNGRP','PCURTY']]
T_ZFPSL = ZFPSL[['Source System Basic Data','Original Source Product Code','Product Group GLINT','Currency']]

today = datetime.today().strftime('%Y%m%d')

In [ ]:
CDPAR2['PGROUP'] = CDPAR2['PGROUP'].astype(str).str.zfill(3)
LNPAR2['PLNGRP'] = LNPAR2['PLNGRP'].astype(str).str.zfill(3)
DDPAR2['PGRCOD'] = DDPAR2['PGRCOD'].astype(str).str.zfill(3)
ZFPSL['Product Group GLINT'] = ZFPSL['Product Group GLINT'].astype(str).str.zfill(3)

In [ ]:
warnings.filterwarnings("ignore", category=pd.errors.SettingWithCopyWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

T_ZFPSL = ZFPSL[['Source System Basic Data', 'Original Source Product Code',
                 'Product Group GLINT', 'Currency']].copy()

def clean_str(df, cols):
    df = df.copy()
    for c in cols:
        if c in df.columns:
            df[c] = df[c].astype('object')
            df[c] = df[c].fillna('')
            df[c] = df[c].astype(str).str.strip()
    return df

sources = [
    (CDPAR2[['PTYPE','PGROUP','PCURTY']], 'BNCD',
     ['PTYPE','PGROUP','PCURTY'],
     ['Original Source Product Code','Product Group GLINT','Currency']),

    (LNPAR2[['PTYPE','PLNGRP','PCURTY']], 'BNLN',
     ['PTYPE','PLNGRP','PCURTY'],
     ['Original Source Product Code','Product Group GLINT','Currency']),

    (DDPAR2[['SCCODE','PGRCOD','DP2CUR']], 'BNDD',
     ['SCCODE','PGRCOD','DP2CUR'],
     ['Original Source Product Code','Product Group GLINT','Currency'])
]

unmatch_dict = {}
summary_data = []

# === PROSES PER DATAFRAME ===
for df, label, left_cols, right_cols in sources:

    print(f"🔄 Processing {label} ...")

    df = clean_str(df, left_cols)

    # ======================================================
    # ✅ PAD KIRI: PASTIKAN MENJADI 3 DIGIT (001, 045, 900)
    # ======================================================
    pad_col = left_cols[1]       # PGROUP / PLNGRP / PGRCOD
    df[pad_col] = df[pad_col].str.zfill(3)

    # ======================================================
    # ✅ EXCLUDE NILAI "000" AGAR TIDAK MASUK UNMATCH
    # ======================================================
    before = len(df)
    df = df[df[pad_col] != '000']
    after = len(df)
    print(f"➡️ Filter '000' pada {pad_col}: {before} → {after} baris")

    # ======================================================
    # CLEAN ZFPSL
    # ======================================================
    T_ZFPSL_clean = clean_str(T_ZFPSL, right_cols + ['Source System Basic Data'])
    filtered_zfpsl = T_ZFPSL_clean[T_ZFPSL_clean['Source System Basic Data'] == label]

    if filtered_zfpsl.empty:
        print(f"⚠️ Tidak ada data ZFPSL untuk {label}, menggunakan semua ZFPSL.")
        filtered_zfpsl = T_ZFPSL_clean.copy()

    merged = df.merge(
        filtered_zfpsl,
        how='left',
        left_on=left_cols,
        right_on=right_cols,
        indicator=True
    )

    unmatch = merged.loc[merged['_merge'] == 'left_only', left_cols].drop_duplicates().copy()
    unmatch.insert(0, 'Source System Basic Data', label)
    unmatch = unmatch[['Source System Basic Data'] + left_cols]

    unmatch_dict[label] = unmatch
    summary_data.append({
        'Source System': label,
        'Total Data': len(df),
        'Unmatch Count': len(unmatch),
        'Matched Count': len(df) - len(unmatch)
    })

    print(f"➡️ {label}: total {len(df)}, unmatch {len(unmatch)}")


# === GABUNGKAN SEMUA UNMATCH ===
Unmatch_All = pd.concat(unmatch_dict.values(), ignore_index=True)
Summary = pd.DataFrame(summary_data)

# CLEAN SHEET MASING-MASING
BNCD = clean_str(CDPAR2[['PTYPE','PGROUP','PCURTY']], ['PTYPE','PGROUP','PCURTY'])
BNLN = clean_str(LNPAR2[['PTYPE','PLNGRP','PCURTY']], ['PTYPE','PLNGRP','PCURTY'])
BNDD = clean_str(DDPAR2[['SCCODE','PGRCOD','DP2CUR']], ['SCCODE','PGRCOD','DP2CUR'])
ZFPSL_sheet = T_ZFPSL.copy()

output_file = f'{today}_COMPARE PAR DAN ZFPSL_PRDCTGRP.xlsx'
with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
    Summary.to_excel(writer, sheet_name='SUMMARY', index=False)
    Unmatch_All.to_excel(writer, sheet_name='Unmatch_All', index=False)
    unmatch_dict['BNCD'].to_excel(writer, sheet_name='Unmatch_BNCD', index=False)
    unmatch_dict['BNLN'].to_excel(writer, sheet_name='Unmatch_BNLN', index=False)
    unmatch_dict['BNDD'].to_excel(writer, sheet_name='Unmatch_BNDD', index=False)
    BNCD.to_excel(writer, sheet_name='BNCD', index=False)
    BNLN.to_excel(writer, sheet_name='BNLN', index=False)
    BNDD.to_excel(writer, sheet_name='BNDD', index=False)
    ZFPSL_sheet.to_excel(writer, sheet_name='ZFPSL', index=False)

# === AUTOFIT ===
wb = load_workbook(output_file)
for sheet_name in wb.sheetnames:
    ws = wb[sheet_name]
    for col in ws.columns:
        max_length = 0
        col_letter = get_column_letter(col[0].column)
        for cell in col:
            if cell.value:
                max_length = max(max_length, len(str(cell.value)))
        ws.column_dimensions[col_letter].width = max_length + 2
wb.save(output_file)

print(f"\n✅ File hasil akhir disimpan dan diformat: {output_file}")


🔄 Processing BNCD ...
➡️ Filter '000' pada PGROUP: 352 → 352 baris
➡️ BNCD: total 352, unmatch 0
🔄 Processing BNLN ...
➡️ Filter '000' pada PLNGRP: 1289 → 1267 baris
➡️ BNLN: total 1267, unmatch 0
🔄 Processing BNDD ...
➡️ Filter '000' pada PGRCOD: 664 → 660 baris
➡️ BNDD: total 660, unmatch 0

✅ File hasil akhir disimpan dan diformat: 20260521_COMPARE PAR DAN ZFPSL_PRDCTGRP.xlsx
